# Publication Explorer

Run this notebook from top to bottom to inspect the local governed tables in this repo.

This explorer now supports five local scopes:
- `seed`: the preserved seed-archetype package under `platform/fabric/publication/seed`
- `current`: the canonical approved representative package under `platform/fabric/publication/current`
- `candidate`: the candidate public package under `_bmad-output/test-artifacts/fabric-candidate/current`
- `internal_controls`: the Story 5.2 rule/state tables under `_bmad-output/test-artifacts/fabric-candidate/internal-controls/current`
- `representative_controls`: the Story 5.3 plan/rule tables under `_bmad-output/test-artifacts/fabric-representative/internal-controls/current`

Data Wrangler tip:
- After running a cell that shows a DataFrame such as `dim_product` or `fact_inventory_inbound`, use `Open in Data Wrangler` from the notebook output.
- You can also call `show_table("fact_inventory_inbound", scope="current", limit=25)`, `show_table("dim_account", scope="seed", limit=25)`, or `show_table("plan_account_population", scope="representative_controls", limit=25)` in a new cell.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "package.json").exists() and (candidate / "platform" / "fabric").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root from the current working directory.")


DEFAULT_PREVIEW_ROWS = 50

repo_root = find_repo_root(Path.cwd().resolve())
seed_dir = repo_root / "platform" / "fabric" / "publication" / "seed"
current_dir = repo_root / "platform" / "fabric" / "publication" / "current"
candidate_dir = repo_root / "_bmad-output" / "test-artifacts" / "fabric-candidate" / "current"
internal_controls_dir = (
    repo_root / "_bmad-output" / "test-artifacts" / "fabric-candidate" / "internal-controls" / "current"
)
representative_controls_dir = (
    repo_root / "_bmad-output" / "test-artifacts" / "fabric-representative" / "internal-controls" / "current"
)

print(f"Repo root: {repo_root}")
print(f"Seed archetype package: {seed_dir}")
print(f"Approved current package: {current_dir}")
print(f"Candidate package: {candidate_dir}")
print(f"Internal controls: {internal_controls_dir}")
print(f"Representative controls: {representative_controls_dir}")

In [ ]:
def load_scope_tables(base_dir: Path) -> dict[str, pd.DataFrame]:
    if not base_dir.exists():
        return {}
    return {
        path.stem: pd.read_json(path)
        for path in sorted(base_dir.glob("*.json"))
    }


def classify_table_kind(table_name: str) -> str:
    if table_name.startswith("dim_"):
        return "dimension"
    if table_name.startswith("fact_"):
        return "fact"
    if table_name.startswith("rule_"):
        return "rule"
    if table_name.startswith("state_"):
        return "state"
    if table_name.startswith("plan_"):
        return "plan"
    if table_name.startswith("seed_"):
        return "seed"
    return "other"


def preview_table(df: pd.DataFrame, limit: int = DEFAULT_PREVIEW_ROWS, start: int = 0) -> pd.DataFrame:
    if limit is None:
        preview = df.iloc[start:]
    else:
        preview = df.iloc[start:start + max(limit, 0)]
    display(preview)
    if len(preview) < len(df):
        end = start + len(preview)
        print(f"Previewing rows {start}..{max(end - 1, start)} of {len(df)} total rows. Use a higher limit or start offset to inspect more.")
    return preview


def bind_tables(scope_name: str, tables: dict[str, pd.DataFrame]) -> None:
    prefix = "" if scope_name == "current" else f"{scope_name}_"
    for name, df in tables.items():
        globals()[f"{prefix}{name}"] = df


def show_table(table_name: str, scope: str = "current", limit: int = DEFAULT_PREVIEW_ROWS, start: int = 0) -> pd.DataFrame:
    scope_tables = scope_tables_map.get(scope)
    if scope_tables is None:
        raise KeyError(f"Unknown scope: {scope}")
    if table_name not in scope_tables:
        available = ", ".join(sorted(scope_tables)) or "<none>"
        raise KeyError(f"Table '{table_name}' not found in scope '{scope}'. Available: {available}")
    df = scope_tables[table_name]
    print(f"Showing {scope}.{table_name} -> {len(df)} rows x {len(df.columns)} columns")
    preview_table(df, limit=limit, start=start)
    return df


seed_tables = load_scope_tables(seed_dir)
current_tables = load_scope_tables(current_dir)
candidate_tables = load_scope_tables(candidate_dir)
internal_control_tables = load_scope_tables(internal_controls_dir)
representative_control_tables = load_scope_tables(representative_controls_dir)

scope_tables_map = {
    "seed": seed_tables,
    "current": current_tables,
    "candidate": candidate_tables,
    "internal_controls": internal_control_tables,
    "representative_controls": representative_control_tables,
}

bind_tables("seed", seed_tables)
bind_tables("current", current_tables)
bind_tables("candidate", candidate_tables)
bind_tables("internal_controls", internal_control_tables)
bind_tables("representative_controls", representative_control_tables)

summary_records = []
for scope_name, tables in scope_tables_map.items():
    for table_name, df in tables.items():
        summary_records.append(
            {
                "scope": scope_name,
                "table": table_name,
                "kind": classify_table_kind(table_name),
                "rows": len(df),
                "columns": len(df.columns),
            }
        )

if not summary_records:
    raise RuntimeError("No local publication or internal-control tables were found.")

scope_summary = pd.DataFrame(summary_records).groupby(["scope", "kind"], as_index=False).agg(
    table_count=("table", "count"),
    total_rows=("rows", "sum"),
)
table_summary = pd.DataFrame(summary_records).sort_values(["scope", "kind", "table"]).reset_index(drop=True)

print("Available helper:")
print("- show_table('dim_product')")
print("- show_table('dim_account', scope='seed', limit=25)")
print("- show_table('fact_inventory_inbound', scope='current', limit=25)")
print("- show_table('rule_inventory_inbound', scope='internal_controls', limit=25)")
print("- show_table('plan_account_population', scope='representative_controls', limit=25)")

display(scope_summary)
display(table_summary)

In [ ]:
current_dimensions = {
    name: df
    for name, df in current_tables.items()
    if name.startswith("dim_")
}
display(pd.DataFrame({
    "dimension_table": list(sorted(current_dimensions.keys())),
    "rows": [len(current_dimensions[name]) for name in sorted(current_dimensions.keys())],
}))

preview_table(current_dimensions["dim_product"], limit=15)
preview_table(current_dimensions["dim_product_family"], limit=15)
preview_table(current_dimensions["dim_account"], limit=15)
preview_table(current_dimensions["dim_inventory_location"], limit=15)


In [ ]:
current_facts = {
    name: df
    for name, df in current_tables.items()
    if name.startswith("fact_")
}
display(pd.DataFrame({
    "fact_table": list(sorted(current_facts.keys())),
    "rows": [len(current_facts[name]) for name in sorted(current_facts.keys())],
}))

preview_table(current_facts["fact_demand_line"], limit=20)
preview_table(current_facts["fact_inventory_inbound"], limit=20)
preview_table(current_facts["fact_inventory_snapshot"], limit=20)


In [ ]:
for scope_name, tables in [
    ("seed", seed_tables),
    ("candidate", candidate_tables),
    ("internal_controls", internal_control_tables),
    ("representative_controls", representative_control_tables),
]:
    if tables:
        display(pd.DataFrame({
            f"{scope_name}_table": list(sorted(tables.keys())),
            "rows": [len(tables[name]) for name in sorted(tables.keys())],
        }))
    else:
        display(pd.DataFrame([{"note": f"No tables were found for scope '{scope_name}'."}]))

In [ ]:
if seed_tables:
    preview_table(seed_tables["dim_account"], limit=15)
    preview_table(seed_tables["dim_product"], limit=15)
else:
    display(pd.DataFrame([{"note": "No seed archetype package was found to display."}]))

if current_tables:
    preview_table(current_tables["dim_account"], limit=15)
    preview_table(current_tables["dim_product"], limit=15)
    preview_table(current_tables["fact_demand_line"], limit=20)
    preview_table(current_tables["fact_inventory_snapshot"], limit=20)
else:
    display(pd.DataFrame([{"note": "No canonical current package was found to display."}]))

if representative_control_tables:
    preview_table(representative_control_tables["plan_account_population"], limit=15)
    preview_table(representative_control_tables["plan_product_population"], limit=15)
    preview_table(representative_control_tables["plan_account_product_eligibility"], limit=20)
    preview_table(representative_control_tables["plan_product_location_stocking"], limit=20)
else:
    display(pd.DataFrame([{"note": "No representative internal tables were found to display."}]))